In [ ]:
#| default_exp character_sheets

# character_sheets

> Generate reference sheet images for each character.
>
> Only runs when the selected renderer supports `reference_images`.
> Each sheet is a single image showing the character in a neutral pose,
> used as a visual anchor for all subsequent panel renders featuring that character.
> Results are cached — already-rendered sheets are skipped on resume.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import asyncio
from pathlib import Path

from rich.console import Console
from rich.progress import Progress, SpinnerColumn, TextColumn

from manhualizer.config import OutputConfig, PipelineConfig
from manhualizer.models import Character, Panel, RenderResult, StoryAnalysis
from manhualizer.prompts import TemplateSet, build_character_sheet_prompt
from manhualizer.render import BaseRenderer

_console = Console()

In [ ]:
#| export
def _sheet_path(output_dir: Path, character: Character, fmt: str) -> Path:
    """Canonical path for a character's reference sheet image."""
    safe_name = character.name.lower().replace(" ", "_")
    return output_dir / f"{safe_name}.{fmt}"

In [ ]:
#| export
def _make_sheet_panel(character: Character, templates: TemplateSet, panel_number: int = 0) -> Panel:
    """Create a synthetic Panel representing a character reference sheet render.

    The visual_prompt is built from the character's reference_image_prompt
    using the active template's character_sheet_template.
    """
    # Use the character's existing reference_image_prompt as the description,
    # augmented with physical description for maximum detail.
    description = character.reference_image_prompt or character.physical_description
    if character.physical_description and character.physical_description not in description:
        description = f"{description}, {character.physical_description}"

    visual_prompt = build_character_sheet_prompt(
        templates, character_description=description
    )
    return Panel(
        panel_number=panel_number,
        scene_id="character_sheets",
        characters_present=[character.name],
        location="neutral background",
        action_description=f"Character reference sheet for {character.name}",
        visual_prompt=visual_prompt,
    )

In [ ]:
#| export
async def _generate_one(
    character: Character,
    renderer: BaseRenderer,
    output_dir: Path,
    output_cfg: OutputConfig,
    templates: TemplateSet,
    resume: bool,
    semaphore: asyncio.Semaphore,
    panel_number: int,
) -> tuple[str, Path]:
    """Generate (or resume) a single character sheet. Returns (character_name, image_path)."""
    out_path = _sheet_path(output_dir, character, output_cfg.format)

    if resume and out_path.exists():
        _console.print(f"[dim]character_sheets: skipping {character.name} (exists)[/dim]")
        return character.name, out_path

    sheet_panel = _make_sheet_panel(character, templates, panel_number)
    async with semaphore:
        result: RenderResult = await renderer.render_async(
            sheet_panel, output_dir, output_cfg, reference_images=None
        )
    # Rename from panel_XXXX.fmt to <character_name>.fmt
    if result.image_path != out_path:
        result.image_path.rename(out_path)
    return character.name, out_path

In [ ]:
#| export
async def generate_character_sheets_async(
    analysis: StoryAnalysis,
    renderer: BaseRenderer,
    output_dir: Path,
    templates: TemplateSet,
    config: PipelineConfig,
    concurrency: int = 2,
) -> dict[str, Path]:
    """Generate reference sheet images for all characters in the analysis.

    Only called when `renderer.model_spec.capabilities.reference_images` is True.
    Sheets are rendered concurrently up to `concurrency` at a time.

    Args:
        analysis: StoryAnalysis with the list of characters to render.
        renderer: Active renderer (must support reference_images).
        output_dir: Directory to write sheet images into.
        templates: Active TemplateSet (provides style + character_sheet_template).
        config: PipelineConfig (uses output and resume settings).
        concurrency: Max parallel renders.

    Returns:
        Dict mapping character name → image path.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_cfg = config.output

    _console.print(f"[bold]character_sheets:[/bold] generating {len(analysis.characters)} sheet(s)")

    sem = asyncio.Semaphore(concurrency)
    tasks = [
        _generate_one(char, renderer, output_dir, output_cfg, templates,
                      config.resume, sem, i)
        for i, char in enumerate(analysis.characters)
    ]

    results = await asyncio.gather(*tasks)
    sheets: dict[str, Path] = dict(results)

    _console.print(f"[green]character_sheets: {len(sheets)} sheet(s) ready[/green]")
    for name, path in sheets.items():
        _console.print(f"  {name}: {path}")

    return sheets


def generate_character_sheets(
    analysis: StoryAnalysis,
    renderer: BaseRenderer,
    output_dir: Path,
    templates: TemplateSet,
    config: PipelineConfig,
    concurrency: int = 2,
) -> dict[str, Path]:
    """Sync wrapper around generate_character_sheets_async."""
    return asyncio.run(
        generate_character_sheets_async(
            analysis, renderer, output_dir, templates, config, concurrency
        )
    )

## Tests (no API calls)

In [ ]:
import asyncio, tempfile
from pathlib import Path
from manhualizer.character_sheets import _make_sheet_panel, _sheet_path, generate_character_sheets
from manhualizer.models import Character, RenderResult, StoryAnalysis
from manhualizer.prompts import load_templates
from manhualizer.config import PipelineConfig, OutputConfig, RendererConfig
from manhualizer.render import BaseRenderer, MODELS

templates = load_templates("default")

wei = Character(
    name="Wei Chen",
    physical_description="Tall young man with short black hair",
    personality="Determined",
    reference_image_prompt="young Chinese man, short black hair, determined expression",
)
mei = Character(
    name="Mei Lin",
    physical_description="Slender young woman with long dark hair",
    personality="Wise",
    reference_image_prompt="young Chinese woman, long dark hair, wise expression",
)
analysis = StoryAnalysis(
    title="Test", synopsis="A test",
    characters=[wei, mei], locations=[], source_chunks=["chunk"],
)

# Sheet path naming
with tempfile.TemporaryDirectory() as tmp:
    p = _sheet_path(Path(tmp), wei, "png")
    assert p.name == "wei_chen.png"

# Panel construction
panel = _make_sheet_panel(wei, templates, panel_number=0)
assert panel.scene_id == "character_sheets"
assert "Wei Chen" in panel.characters_present
assert "manhua" in panel.visual_prompt
assert "young Chinese man" in panel.visual_prompt

print("Sheet panel construction OK")

In [ ]:
# Full generate test with stub renderer
import asyncio, tempfile
from pathlib import Path
from manhualizer.render import BaseRenderer, MODELS
from manhualizer.config import PipelineConfig, RendererConfig
from manhualizer.models import Panel, RenderResult

class StubRenderer(BaseRenderer):
    async def render_async(self, panel, output_dir, output_cfg, reference_images=None):
        path = output_dir / f"panel_{panel.panel_number:04d}.{output_cfg.format}"
        path.write_bytes(b"stub_image")
        return RenderResult(
            panel_number=panel.panel_number,
            image_path=path,
            backend_used="stub",
            prompt_used=panel.visual_prompt,
        )

renderer = StubRenderer(MODELS["nanobanana"], RendererConfig())
cfg = PipelineConfig(resume=False)

with tempfile.TemporaryDirectory() as tmp:
    sheets = generate_character_sheets(
        analysis, renderer, Path(tmp) / "sheets", templates, cfg
    )
    assert len(sheets) == 2
    assert "Wei Chen" in sheets
    assert "Mei Lin" in sheets
    assert sheets["Wei Chen"].exists()
    assert sheets["Wei Chen"].name == "wei_chen.png"
    assert sheets["Mei Lin"].name == "mei_lin.png"

    # Resume: pre-existing sheets are skipped
    cfg_resume = PipelineConfig(resume=True)
    sheets2 = generate_character_sheets(
        analysis, renderer, Path(tmp) / "sheets", templates, cfg_resume
    )
    assert len(sheets2) == 2

print("Character sheet generation OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()